<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/redes-computadores-3/01_enrutamiento_ls_dv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Estado de enlace vs. vector distancia: Dijkstra y Bellman-Ford

**Redes de Computadores III** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Implementar Dijkstra (base de OSPF) y construir la tabla de reenvío.
- Simular vector distancia (base de RIP) ronda a ronda.
- Observar la convergencia y el problema de la cuenta al infinito.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
import math
import networkx as nx, matplotlib.pyplot as plt

In [ ]:
import heapq
red = {  # enlaces bidireccionales con costo
    ("A", "B"): 2, ("A", "C"): 5, ("B", "C"): 1, ("B", "D"): 4,
    ("C", "D"): 1, ("C", "E"): 6, ("D", "E"): 2, ("D", "F"): 7, ("E", "F"): 1,
}
def vecinos(red):
    v = {}
    for (a, b), c in red.items():
        v.setdefault(a, {})[b] = c; v.setdefault(b, {})[a] = c
    return v

def dijkstra(red, origen):
    V = vecinos(red); dist = {n: math.inf for n in V}; prev = {}
    dist[origen] = 0; cola = [(0, origen)]
    while cola:
        d, u = heapq.heappop(cola)
        if d > dist[u]: continue
        for w, c in V[u].items():
            if d + c < dist[w]:
                dist[w] = d + c; prev[w] = u; heapq.heappush(cola, (dist[w], w))
    return dist, prev

def vector_distancia(red, max_rondas=50, tablas=None, verbose=False):
    """Bellman-Ford distribuido síncrono: en cada ronda cada router envía su vector a sus vecinos."""
    V = vecinos(red); nodos = sorted(V)
    if tablas is None:
        tablas = {n: {m: (0 if m == n else math.inf) for m in nodos} for n in nodos}
    for ronda in range(1, max_rondas + 1):
        nuevas = {}
        for n in nodos:
            nuevas[n] = {m: (0 if m == n else min([V[n][w] + tablas[w][m] for w in V[n]], default=math.inf)) for m in nodos}
        if nuevas == tablas:
            return tablas, ronda - 1
        tablas = nuevas
        if verbose: print(f"ronda {ronda}: A ->", tablas["A"])
    return tablas, max_rondas

G = nx.Graph(); [G.add_edge(a, b, weight=c) for (a, b), c in red.items()]
pos = nx.spring_layout(G, seed=3)
nx.draw(G, pos, with_labels=True, node_color="#dcefe4", node_size=700)
nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "weight")); plt.show()

## 1. Estado de enlace (OSPF)
Cada router conoce **toda** la topología (inundación de LSAs) y ejecuta Dijkstra localmente.

In [ ]:
dist, prev = dijkstra(red, "A")
def siguiente_salto(prev, origen, destino):
    while prev.get(destino) != origen: destino = prev[destino]
    return destino
print("Destino  Costo  Siguiente salto")
for n in sorted(dist):
    if n != "A": print(f"{n:8s} {dist[n]:5}  {siguiente_salto(prev, 'A', n)}")

## 2. Vector distancia (RIP)
Cada router solo conoce a sus vecinos e intercambia su vector de distancias: $D_x(y) = \min_v \{c(x,v) + D_v(y)\}$.

In [ ]:
tablas, rondas = vector_distancia(red, verbose=True)
print('Convergió en', rondas, 'rondas')

**Ejercicio 1.** Costo del camino más corto de A a F → `resp_1`.

**Ejercicio 2.** Número de rondas que tarda vector distancia en converger (valor que devuelve la función) → `resp_2`.

In [ ]:
resp_1 = None
resp_2 = None

## 3. Cuenta al infinito
Partiendo de las tablas convergidas, el enlace **E–F** se cae. Observe cómo las distancias hacia F crecen lentamente.

In [ ]:
red_caida = {k: v for k, v in red.items() if k != ("E", "F")}
_, r = vector_distancia(red_caida, tablas=tablas, verbose=True)
print("Rondas hasta reconverger:", r)

**Ejercicio 3.** Ahora elimine también el enlace **D–F** (F queda aislado). ¿Qué ocurre? Use `max_rondas=30`. Explique cómo *split horizon* y *poison reverse* mitigan el problema, e implemente *split horizon*.

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 7)
verificar('Ejercicio 2', resp_2, 5)